# 🚀 Keedio Financial AI — Exploratory Data Analysis

> Automated EDA for the monthly project closing dataset.
> **Goal:** Understand revenue patterns, cost structure, margins, and anomalies before forecasting.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.transformation.pipeline import DataPipeline
from src.exploration.eda import EDA
from src.forecasting.prophet_model import FinancialForecaster
from src.anomaly_detection.detector import AnomalyDetector
from src.visualization.charts import ExecutiveCharts

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

In [ ]:
pipeline = DataPipeline('../data/raw/KEEDIO_Cierre_Mensual.csv')
df = pipeline.run()
monthly = pipeline.get_monthly_aggregate()
print(f'Shape: {df.shape}')
print(f'Date range: {df["ds"].min()} → {df["ds"].max()}')
print(f'Projects: {df["Proyecto"].nunique()}')
print(f'Clients: {df["Cliente"].nunique()}')

In [ ]:
df.head(10)

In [ ]:
eda = EDA(df)
eda.summary()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
num_cols = ['Ingresos', 'CosteEquipo', 'MargenBruto', 'HorasFacturadas']
for ax, col in zip(axes.flatten(), num_cols):
    sns.histplot(df[col], bins=30, kde=True, ax=ax, color='#2563EB')
    ax.axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean: {df[col].mean():,.0f}')
    ax.axvline(df[col].median(), color='green', linestyle='--', label=f'Median: {df[col].median():,.0f}')
    ax.set_title(f'Distribution: {col}')
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(monthly['ds'], monthly['Ingresos'], marker='o', linewidth=2, color='#2563EB', label='Revenue')
ax.plot(monthly['ds'], monthly['CosteEquipo'], marker='s', linewidth=2, color='#EF4444', label='Cost')
ax.plot(monthly['ds'], monthly['MargenBruto'], marker='^', linewidth=2, color='#10B981', label='Gross Margin')
ax.fill_between(monthly['ds'], monthly['CosteEquipo'], monthly['Ingresos'], alpha=0.1, color='#2563EB')
ax.set_title('Monthly Revenue, Cost & Margin Trend', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.show()

In [ ]:
pivot = df.pivot_table(index='ds', columns='Servicio', values='Ingresos', aggfunc='sum')
fig, ax = plt.subplots(figsize=(14, 6))
pivot.plot(kind='area', ax=ax, alpha=0.7, color=['#2563EB', '#10B981', '#8B5CF6'])
ax.set_title('Revenue by Service Line', fontsize=14, fontweight='bold')
ax.legend(title='Service')
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.show()

In [ ]:
forecaster = FinancialForecaster(monthly, target='Ingresos')
forecaster.train()
forecaster.forecast_future(periods=90)
print('Model Metrics:', forecaster.metrics)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))
train_df = forecaster.prepare()
ax.plot(train_df['ds'], train_df['y'], marker='o', linewidth=2, color='#2563EB', label='Historical')
ax.plot(forecaster.forecast['ds'], forecaster.forecast['yhat'], linestyle='--', linewidth=2, color='#EF4444', label='Forecast')
ax.fill_between(forecaster.forecast['ds'], forecaster.forecast['yhat_lower'], forecaster.forecast['yhat_upper'], alpha=0.2, color='#EF4444', label='80% CI')
ax.axvline(x=train_df['ds'].iloc[-1], color='gray', linestyle=':', alpha=0.7)
ax.set_title('Revenue Forecast — Prophet (30/60/90 days)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.show()

In [ ]:
forecaster.get_insights()

In [ ]:
detector = AnomalyDetector(df)
anomalies = detector.detect_all(monthly)
print(f'Z-score anomalies: {len(anomalies["zscore"])}')
print(f'Isolation Forest anomalies: {len(anomalies["isolation_forest"])}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
targets = ['Ingresos', 'CosteEquipo', 'MargenBruto', 'HorasFacturadas']
for ax, col in zip(axes.flatten(), targets):
    ax.scatter(df['ds'], df[col], alpha=0.6, color='#2563EB', label='Normal')
    ax.scatter(anomalies['isolation_forest']['ds'], anomalies['isolation_forest'][col], 
               color='#EF4444', s=100, marker='x', label='Anomaly')
    ax.set_title(f'Anomalies — {col}')
    ax.legend()
    ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
scenarios = [
    {'name': 'Base Case', 'adjustments': {}},
    {'name': 'Growth +10%', 'adjustments': {'growth': 10}},
    {'name': 'Decline -15%', 'adjustments': {'decline': 15}},
]
forecaster.scenario_analysis(scenarios)

In [ ]:
charts = ExecutiveCharts(df, monthly)
charts.kpi_dashboard()
plt.show()

## Key Takeaways

1. Revenue shows clear seasonality with Q4 peaks
2. Profit margins are healthy (30-40% range)
3. IA projects generate highest margins
4. Anomalies cluster around months with extreme project swings
5. Prophet forecasts stable growth with manageable risk

---
*Report generated by Keedio Financial AI Forecasting System*